# Notebook 05: Official Locked Test-Set Evaluation & Multimodal Analysis (WLASL-100)

**Project Milestone & Evaluation Protocol:**
Throughout Notebooks 01 through 04c, the **test split (230 clips across 20 unseen signers)** was strictly locked:
- **Notebook 03 (Visual ResNet50 Baselines):** Top-1 = 11.93% | Top-5 = 22.94% (plateaued due to signer & room background shortcuts).
- **Notebook 04 (Multimodal ResNet50 + Language):** Top-1 = 11.01% | Top-5 = 25.69% (limited by noisy full-frame ImageNet pooling).
- **Notebook 04b (53-Keypoint Pose & Hands Upgrade):** Top-1 = 41.28% | Top-5 = 68.81% (3.7x accuracy jump, background shortcut eliminated).
- **Notebook 04c (Kinematic Boost & Multi-Inductive Ensembling):** Top-1 = 43.12% | Top-5 = 72.48% | Macro-F1 = 0.4090 (winning ensemble `ensemble_equal_three`).

**What Notebook 05 Executes Today:**
1. **Evaluates ALL finalized models** on the unseen **230-clip test set**:
   - Random Chance & Majority Class baselines
   - ResNet50 Visual Transformer (`visual_transformer_attn`)
   - ResNet50 Multimodal Contrastive (`mm_contrastive_prompt`)
   - Pose Transformer Only (`pose_transformer_only`)
   - Multimodal Pose + Language (`mm_pose_language`)
   - Boosted Pose Transformer + Cosine (`pose_transformer_cosine`)
   - Multi-Inductive-Bias Winning Ensemble (`ensemble_equal_three`)
2. **Generates Official Metrics:** Top-1, Top-5, Top-10, Macro-F1, Weighted-F1, Video->Text Recall@1/5/10, MRR.
3. **Produces Publication-Ready Visualizations:**
   - Evolution Comparison Bar Chart (`05_final_test_comparison.png`)
   - 100-Class Confusion Matrix Heatmap & Top-5 Confused Pairs (`05_confusion_matrix.png`)
   - Best vs. Most Challenging Classes Breakdown (`05_class_performance.png`)
   - Multimodal t-SNE 2D Kinematic-Language Joint Space Alignment (`05_tsne_multimodal_alignment.png`)
4. **Exports Production Artifacts:**
   - Full evaluation report (`experiments/test_evaluation_report.json`)
   - Updates `experiments/results_summary.csv` with official test scores
   - Packages winning model & ensemble weights into `checkpoints/production_model/` for deployment.


## 1. Environment Setup & Google Drive Mount


In [ ]:
import json
import math
import os
import random
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.manifold import TSNE

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    print("Google Drive mounted at /content/drive.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | Device: {DEVICE}")


## 2. Load Locked Test Features & Model Checkpoints


In [ ]:
import json
from pathlib import Path

if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/ASL-Recognition/voxel51_wlasl100")
else:
    DRIVE_ROOT = Path("./voxel51_wlasl100")

META_DIR = DRIVE_ROOT / "metadata"
FEAT_DIR = DRIVE_ROOT / "features"
CKPT_DIR = DRIVE_ROOT / "checkpoints"
EXP_DIR = DRIVE_ROOT / "experiments"
PLOTS_DIR = DRIVE_ROOT / "reports" / "plots"
PROD_DIR = CKPT_DIR / "production_model"

for d in (PLOTS_DIR, PROD_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Load class mappings
with open(META_DIR / "class_to_id.json") as f:
    class_to_id = json.load(f)
with open(META_DIR / "id_to_class.json") as f:
    id_to_class = {int(k): v for k, v in json.load(f).items()}

NUM_CLASSES = len(class_to_id)
gloss_names = [id_to_class[i] for i in range(NUM_CLASSES)]

# Load test set features (LOCKED TEST SET)
P_test = np.load(FEAT_DIR / "P_test.npy").astype(np.float32)
y_test = np.load(FEAT_DIR / "y_test.npy").astype(np.int64)
X_test = np.load(FEAT_DIR / "X_test.npy").astype(np.float32)

POSE_DIM = P_test.shape[-1]
NUM_FRAMES = P_test.shape[1]

# Load training labels to compute majority baseline
y_train = np.load(FEAT_DIR / "y_train.npy").astype(np.int64)

# Load text vocabulary embeddings
prompt_pt = torch.load(FEAT_DIR / "text_embeddings_prompt.pt", map_location="cpu", weights_only=False)
vocab_prompt = prompt_pt["text_embeddings"].float().to(DEVICE)

print(f"Locked Test Set Loaded:")
print(f"  P_test      : shape={P_test.shape}, dtype={P_test.dtype}")
print(f"  X_test      : shape={X_test.shape}, dtype={X_test.dtype}")
print(f"  y_test      : {len(y_test)} clips across {len(np.unique(y_test))} unique classes")
print(f"  vocab_prompt: shape={vocab_prompt.shape}")


## 3. Architecture Definitions & Checkpoint Loading Engine


In [ ]:
class CosineClassifier(nn.Module):
    def __init__(self, in_features: int, num_classes: int, init_scale: float = 16.0):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.logit_scale = nn.Parameter(
            torch.tensor(math.log(max(init_scale, 1.0)), dtype=torch.float32)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_norm = F.normalize(x, p=2, dim=-1)
        w_norm = F.normalize(self.weight, p=2, dim=-1)
        scale = self.logit_scale.exp().clamp(min=1.0, max=50.0)
        return scale * F.linear(x_norm, w_norm)


class TemporalAttentionPool(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.proj = nn.Linear(dim, 1)

    def forward(self, seq: torch.Tensor) -> torch.Tensor:
        return (seq * torch.softmax(self.proj(seq), dim=1)).sum(dim=1)


class TemporalTransformerEncoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        embed_dim: int = 192,
        num_layers: int = 2,
        num_heads: int = 4,
        dim_feedforward: int = 384,
        dropout: float = 0.35,
        max_seq_len: int = 128,
    ):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        pe = torch.zeros(1, max_seq_len, embed_dim)
        pos = torch.arange(0, max_seq_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, embed_dim, 2, dtype=torch.float32) * (-math.log(10000.0) / embed_dim)
        )
        pe[0, :, 0::2] = torch.sin(pos * div)
        pe[0, :, 1::2] = torch.cos(pos * div)
        self.register_buffer("pos_encoding", pe, persistent=False)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(embed_dim)
        self.attn_pool = TemporalAttentionPool(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        t_len = x.size(1)
        h = self.input_proj(x) + self.pos_encoding[:, :t_len, :]
        return self.attn_pool(self.norm(self.encoder(h)))


class TemporalBiLSTMEncoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 96,
        num_layers: int = 2,
        dropout: float = 0.35,
    ):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.norm = nn.LayerNorm(hidden_dim * 2)
        self.attn_pool = TemporalAttentionPool(hidden_dim * 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.input_proj(x)
        seq, _ = self.lstm(h)
        return self.attn_pool(self.norm(seq))


class TextProjectionHead(nn.Module):
    def __init__(self, input_dim: int = 384, embed_dim: int = 192, dropout: float = 0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(x), p=2, dim=-1)


class PoseBoostedASLModel(nn.Module):
    def __init__(
        self,
        num_classes: int = 100,
        *,
        pose_dim: int = 349,
        text_dim: int = 384,
        hidden_dim: int = 192,
        embed_dim: int = 192,
        num_layers: int = 2,
        num_heads: int = 4,
        dropout: float = 0.35,
        backbone: str = "transformer",
        mode: str = "pose_only",
        use_cosine_classifier: bool = True,
        fusion_alpha: float = 0.5,
        init_temperature: float = 0.1,
    ):
        super().__init__()
        self.mode = mode
        self.backbone = backbone
        self.use_cosine_classifier = use_cosine_classifier
        self.fusion_alpha = fusion_alpha

        if backbone == "bilstm":
            self.pose_encoder = TemporalBiLSTMEncoder(
                input_dim=pose_dim,
                hidden_dim=hidden_dim // 2,
                num_layers=num_layers,
                dropout=dropout,
            )
        else:
            self.pose_encoder = TemporalTransformerEncoder(
                input_dim=pose_dim,
                embed_dim=hidden_dim,
                num_layers=num_layers,
                num_heads=num_heads,
                dim_feedforward=hidden_dim * 2,
                dropout=dropout,
            )

        self.motion_projection = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )

        if use_cosine_classifier:
            self.classifier_head = nn.Sequential(
                nn.Dropout(dropout),
                CosineClassifier(hidden_dim, num_classes),
            )
        else:
            self.classifier_head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, num_classes),
            )

        if mode == "pose_language":
            self.text_projection = TextProjectionHead(text_dim, embed_dim, dropout=0.15)
            self.logit_scale = nn.Parameter(
                torch.tensor(math.log(1.0 / max(init_temperature, 1e-3)), dtype=torch.float32)
            )
            self.lang_fusion_gate = nn.Sequential(
                nn.Linear(embed_dim * 2, embed_dim),
                nn.Sigmoid(),
            )
            if use_cosine_classifier:
                self.fused_head = nn.Sequential(
                    nn.Dropout(dropout),
                    CosineClassifier(embed_dim, num_classes),
                )
            else:
                self.fused_head = nn.Sequential(
                    nn.Dropout(dropout),
                    nn.Linear(embed_dim, num_classes),
                )
        else:
            self.text_projection = None
            self.register_parameter("logit_scale", None)
            self.lang_fusion_gate = None
            self.fused_head = None

    def forward(
        self,
        pose_inputs: torch.Tensor,
        vocab_text_embeddings: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        repr_vec = self.pose_encoder(pose_inputs)
        if self.mode == "pose_only":
            return {"logits": self.classifier_head(repr_vec), "representation": repr_vec}

        vis_emb = F.normalize(self.motion_projection(repr_vec), p=2, dim=-1)
        txt_emb = self.text_projection(vocab_text_embeddings)
        scale = self.logit_scale.exp().clamp(max=30.0)
        cosine_logits = scale * (vis_emb @ txt_emb.T)
        attn = torch.softmax(cosine_logits, dim=-1)
        lang_ctx = attn @ txt_emb
        gate = self.lang_fusion_gate(torch.cat([vis_emb, lang_ctx], dim=-1))
        fused = gate * vis_emb + (1.0 - gate) * lang_ctx
        cls_logits = self.fused_head(fused) + self.fusion_alpha * cosine_logits

        return {
            "logits": cls_logits,
            "cosine_logits": cosine_logits,
            "visual_embeddings": vis_emb,
            "text_embeddings": txt_emb,
            "logit_scale": scale,
        }


## 4. Run Test Inference Across All Model Generations


In [ ]:
import json

def compute_detailed_test_metrics(
    y_true: np.ndarray,
    scores: np.ndarray,
    top_k_list: Tuple[int, ...] = (1, 5, 10),
) -> Dict[str, Any]:
    y_t = np.asarray(y_true, dtype=np.int64)
    sc = np.asarray(scores, dtype=np.float64)
    y_pred = np.argmax(sc, axis=1)

    top_acc = {}
    for k in top_k_list:
        k_idx = min(k, sc.shape[1])
        top_k_hits = np.any(np.argpartition(-sc, kth=k_idx - 1, axis=1)[:, :k_idx] == y_t[:, None], axis=1)
        top_acc[f"top{k}_accuracy"] = float(np.mean(top_k_hits))

    # Per-class metrics
    class_metrics = {}
    f1_list, prec_list, rec_list = [], [], []
    for cid in range(sc.shape[1]):
        tp = int(np.sum((y_t == cid) & (y_pred == cid)))
        fp = int(np.sum((y_t != cid) & (y_pred == cid)))
        fn = int(np.sum((y_t == cid) & (y_pred != cid)))
        support = int(np.sum(y_t == cid))

        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2.0 * p * r / (p + r)) if (p + r) > 0 else 0.0

        class_metrics[id_to_class[cid]] = {
            "precision": round(p, 4),
            "recall": round(r, 4),
            "f1": round(f1, 4),
            "support": support,
        }
        if support > 0:
            f1_list.append(f1)
            prec_list.append(p)
            rec_list.append(r)

    # Weighted F1
    total_samples = len(y_t)
    weighted_f1 = sum(class_metrics[id_to_class[cid]]["f1"] * class_metrics[id_to_class[cid]]["support"] for cid in range(sc.shape[1])) / total_samples

    return {
        **top_acc,
        "macro_f1": float(np.mean(f1_list)),
        "weighted_f1": float(weighted_f1),
        "macro_precision": float(np.mean(prec_list)),
        "macro_recall": float(np.mean(rec_list)),
        "unique_predicted_classes": int(len(np.unique(y_pred))),
        "per_class": class_metrics,
        "predictions": y_pred.tolist(),
    }


def compute_test_retrieval(sim: np.ndarray, q_lbl: np.ndarray, g_lbl: np.ndarray) -> Dict[str, float]:
    order = np.argsort(-sim, axis=1)
    ranks = np.asarray([int(np.flatnonzero(g_lbl[o] == q_lbl[i])[0]) + 1 for i, o in enumerate(order)], dtype=np.float64)
    return {
        "v2t_mrr": float(np.mean(1.0 / ranks)),
        "v2t_recall@1": float(np.mean(ranks <= 1)),
        "v2t_recall@5": float(np.mean(ranks <= 5)),
        "v2t_recall@10": float(np.mean(ranks <= 10)),
    }


test_results = {}
test_probabilities = {}
test_embeddings = {}

# 1. Random Baseline
rng_eval = np.random.default_rng(SEED)
rand_scores = rng_eval.uniform(0.0, 1.0, size=(len(y_test), NUM_CLASSES))
test_results["random_chance"] = {
    "model_name": "Uniform Random",
    "modality": "none",
    **compute_detailed_test_metrics(y_test, rand_scores),
}

# 2. Majority Class Baseline
maj_class = int(np.bincount(y_train).argmax())
maj_scores = np.zeros((len(y_test), NUM_CLASSES), dtype=np.float64)
maj_scores[:, maj_class] = 1.0
test_results["majority_class"] = {
    "model_name": f"Majority Class ('{id_to_class[maj_class]}')",
    "modality": "none",
    **compute_detailed_test_metrics(y_test, maj_scores),
}

# Helper to infer a PyTorch model on P_test
def infer_pose_model(model: nn.Module, mode: str) -> Tuple[np.ndarray, Optional[np.ndarray], Optional[np.ndarray]]:
    model.eval()
    p_t = torch.from_numpy(P_test).to(DEVICE)
    with torch.inference_mode():
        if mode == "pose_only":
            out = model(p_t)
            vis_emb, txt_emb = None, None
        else:
            out = model(p_t, vocab_text_embeddings=vocab_prompt)
            vis_emb = out["visual_embeddings"].cpu().numpy()
            txt_emb = out["text_embeddings"].cpu().numpy()
        lg = out["logits"].cpu().numpy()
    probs = np.exp(lg - np.max(lg, axis=1, keepdims=True))
    probs /= np.sum(probs, axis=1, keepdims=True)
    return probs, vis_emb, txt_emb


# 3. Notebook 04b: pose_transformer_only
p_ckpt_4b = CKPT_DIR / "pose_transformer_only" / "best_model.pt"
if p_ckpt_4b.is_file():
    ckpt = torch.load(p_ckpt_4b, map_location=DEVICE, weights_only=False)
    m_4b = PoseBoostedASLModel(NUM_CLASSES, pose_dim=349, backbone="transformer", mode="pose_only", use_cosine_classifier=False).to(DEVICE)
    m_4b.load_state_dict(ckpt["model_state_dict"])
    pr, _, _ = infer_pose_model(m_4b, "pose_only")
    test_probabilities["pose_transformer_only"] = pr
    test_results["pose_transformer_only"] = {
        "model_name": "ResNet50-Free Pose Transformer (04b)",
        "modality": "pose_only",
        **compute_detailed_test_metrics(y_test, pr),
    }

# 4. Notebook 04b: mm_pose_language
mm_ckpt_4b = CKPT_DIR / "mm_pose_language" / "best_model.pt"
if mm_ckpt_4b.is_file():
    ckpt = torch.load(mm_ckpt_4b, map_location=DEVICE, weights_only=False)
    m_mm4b = PoseBoostedASLModel(NUM_CLASSES, pose_dim=349, text_dim=384, backbone="transformer", mode="pose_language", use_cosine_classifier=False).to(DEVICE)
    m_mm4b.load_state_dict(ckpt["model_state_dict"])
    pr, ve, te = infer_pose_model(m_mm4b, "pose_language")
    test_probabilities["mm_pose_language"] = pr
    test_embeddings["mm_pose_language"] = (ve, te)
    ret_m = compute_test_retrieval(ve @ te.T, y_test, np.arange(NUM_CLASSES))
    test_results["mm_pose_language"] = {
        "model_name": "MM Pose + Language (04b)",
        "modality": "multimodal_pose",
        **compute_detailed_test_metrics(y_test, pr),
        **ret_m,
    }

# 5. Notebook 04c: pose_transformer_cosine
p_ckpt_4c = CKPT_DIR / "pose_transformer_cosine" / "best_model.pt"
if p_ckpt_4c.is_file():
    ckpt = torch.load(p_ckpt_4c, map_location=DEVICE, weights_only=False)
    m_4c = PoseBoostedASLModel(NUM_CLASSES, pose_dim=349, backbone="transformer", mode="pose_only", use_cosine_classifier=True).to(DEVICE)
    m_4c.load_state_dict(ckpt["model_state_dict"])
    pr, _, _ = infer_pose_model(m_4c, "pose_only")
    test_probabilities["pose_transformer_cosine"] = pr
    test_results["pose_transformer_cosine"] = {
        "model_name": "Pose Transformer + Cosine Head (04c)",
        "modality": "pose_only",
        **compute_detailed_test_metrics(y_test, pr),
    }

# 6. Notebook 04c: pose_bilstm_attn
p_lstm_4c = CKPT_DIR / "pose_bilstm_attn" / "best_model.pt"
if p_lstm_4c.is_file():
    ckpt = torch.load(p_lstm_4c, map_location=DEVICE, weights_only=False)
    m_lstm4c = PoseBoostedASLModel(NUM_CLASSES, pose_dim=349, backbone="bilstm", mode="pose_only", use_cosine_classifier=True).to(DEVICE)
    m_lstm4c.load_state_dict(ckpt["model_state_dict"])
    pr, _, _ = infer_pose_model(m_lstm4c, "pose_only")
    test_probabilities["pose_bilstm_attn"] = pr
    test_results["pose_bilstm_attn"] = {
        "model_name": "Pose BiLSTM + Attn + Cosine (04c)",
        "modality": "pose_only",
        **compute_detailed_test_metrics(y_test, pr),
    }

# 7. Notebook 04c: mm_pose_language_boosted
p_mm_4c = CKPT_DIR / "mm_pose_language_boosted" / "best_model.pt"
if p_mm_4c.is_file():
    ckpt = torch.load(p_mm_4c, map_location=DEVICE, weights_only=False)
    m_mm4c = PoseBoostedASLModel(NUM_CLASSES, pose_dim=349, text_dim=384, backbone="transformer", mode="pose_language", use_cosine_classifier=True).to(DEVICE)
    m_mm4c.load_state_dict(ckpt["model_state_dict"])
    pr, ve, te = infer_pose_model(m_mm4c, "pose_language")
    test_probabilities["mm_pose_language_boosted"] = pr
    test_embeddings["mm_pose_language_boosted"] = (ve, te)
    ret_m = compute_test_retrieval(ve @ te.T, y_test, np.arange(NUM_CLASSES))
    test_results["mm_pose_language_boosted"] = {
        "model_name": "MM Pose + Language Boosted (04c)",
        "modality": "multimodal_pose",
        **compute_detailed_test_metrics(y_test, pr),
        **ret_m,
    }

# 8. Notebook 04c: Winning Ensemble (ensemble_equal_three)
ens_cfg_file = CKPT_DIR / "ensemble_boosted" / "ensemble_config.json"
if ens_cfg_file.is_file():
    with open(ens_cfg_file) as f:
        ens_cfg = json.load(f)
    weights = ens_cfg["weights"]
    p_ens = sum(weights[k] * test_probabilities[k] for k in weights if k in test_probabilities)
    test_probabilities["ensemble_equal_three"] = p_ens
    test_results["ensemble_equal_three"] = {
        "model_name": f"Winning Ensemble ({ens_cfg.get('winning_ensemble_name', 'ensemble_equal_three')})",
        "modality": "ensemble_kinematic_multimodal",
        **compute_detailed_test_metrics(y_test, p_ens),
    }

print("\n" + "=" * 90)
print("OFFICIAL TEST-SET RESULTS ON UNSEEN SIGNERS (N_test=230)")
print("=" * 90)
rows = []
for exp_id, res in test_results.items():
    r10 = f"{res.get('top10_accuracy', 0)*100:5.2f}%" if "top10_accuracy" in res else "-"
    mrr = f"{res.get('v2t_mrr', 0):.4f}" if "v2t_mrr" in res else "-"
    r5 = f"{res.get('v2t_recall@5', 0)*100:5.2f}%" if "v2t_recall@5" in res else "-"
    print(
        f"[{exp_id:26s}] Top-1={res['top1_accuracy']*100:5.2f}% | "
        f"Top-5={res['top5_accuracy']*100:5.2f}% | Top-10={r10} | "
        f"F1={res['macro_f1']:.4f} | MRR={mrr} | Uniq={res['unique_predicted_classes']:3d}/100"
    )
    rows.append({
        "experiment_id": exp_id,
        "model": res["model_name"],
        "modality": res["modality"],
        "split": "test",
        "top1_accuracy": round(res["top1_accuracy"], 4),
        "top5_accuracy": round(res["top5_accuracy"], 4),
        "top10_accuracy": round(res.get("top10_accuracy", np.nan), 4),
        "macro_f1": round(res["macro_f1"], 4),
        "weighted_f1": round(res.get("weighted_f1", np.nan), 4),
        "v2t_recall@5": round(res.get("v2t_recall@5", np.nan), 4),
        "v2t_mrr": round(res.get("v2t_mrr", np.nan), 4),
    })

test_df = pd.DataFrame(rows)


## 5. Visualizations: Evolution Comparison, Confusion Matrix & Error Analysis


In [ ]:
# 1. Progression Comparison Chart
plt.figure(figsize=(12, 5))
sub_df = test_df[~test_df["experiment_id"].isin(["majority_class"])].copy()
x_pos = np.arange(len(sub_df))
bar_w = 0.38

plt.bar(x_pos - bar_w/2, sub_df["top1_accuracy"] * 100.0, width=bar_w, label="Test Top-1 Accuracy (%)", color="#1f77b4")
plt.bar(x_pos + bar_w/2, sub_df["top5_accuracy"] * 100.0, width=bar_w, label="Test Top-5 Accuracy (%)", color="#2ca02c")

for i, r in enumerate(sub_df.itertuples()):
    plt.text(i - bar_w/2, r.top1_accuracy * 100.0 + 1.0, f"{r.top1_accuracy*100:.1f}%", ha="center", fontsize=8, fontweight="bold")
    plt.text(i + bar_w/2, r.top5_accuracy * 100.0 + 1.0, f"{r.top5_accuracy*100:.1f}%", ha="center", fontsize=8)

plt.xticks(x_pos, sub_df["experiment_id"], rotation=30, ha="right", fontsize=9)
plt.ylabel("Accuracy (%)")
plt.title("Model Evolution on Unseen Signers (Locked Test Split, N=230)")
plt.ylim(0, 85)
plt.grid(axis="y", alpha=0.3)
plt.legend(loc="upper left")
plt.tight_layout()
p_comp = PLOTS_DIR / "05_final_test_comparison.png"
plt.savefig(p_comp, dpi=130)
plt.show()

# 2. Confusion Matrix & Top-10 Error Pairs for Winning Model
winning_exp = "ensemble_equal_three" if "ensemble_equal_three" in test_results else "pose_transformer_cosine"
winning_preds = np.array(test_results[winning_exp]["predictions"])

conf_mat = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
for yt, yp in zip(y_test, winning_preds):
    conf_mat[yt, yp] += 1

plt.figure(figsize=(10, 8))
plt.imshow(conf_mat, cmap="Blues", interpolation="nearest")
plt.title(f"100-Class Confusion Matrix: {winning_exp} (Test Split)")
plt.xlabel("Predicted Class ID")
plt.ylabel("Ground Truth Class ID")
plt.colorbar(label="Clip Count")
plt.tight_layout()
p_conf = PLOTS_DIR / "05_confusion_matrix.png"
plt.savefig(p_conf, dpi=130)
plt.show()

# Extract top confused pairs (True != Pred)
confused_pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and conf_mat[i, j] > 0:
            confused_pairs.append({
                "true_gloss": id_to_class[i],
                "pred_gloss": id_to_class[j],
                "error_count": int(conf_mat[i, j]),
            })

confused_df = pd.DataFrame(confused_pairs).sort_values("error_count", ascending=False).reset_index(drop=True)
print("\nTop-10 Most Confused Gloss Pairs on Test Set:")
print(confused_df.head(10).to_string(index=False))

# 3. Best vs. Most Challenging Classes (Per-Class F1)
per_class_df = pd.DataFrame([
    {"gloss": g, **m} for g, m in test_results[winning_exp]["per_class"].items() if m["support"] > 0
]).sort_values("f1", ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top10_c = per_class_df.head(10)
bot10_c = per_class_df.tail(10)

axes[0].barh(top10_c["gloss"][::-1], top10_c["f1"][::-1], color="#2ca02c")
axes[0].set_title(f"Top-10 Strongest Classes (F1) - {winning_exp}")
axes[0].set_xlabel("F1 Score")
axes[0].set_xlim(0, 1.05)

axes[1].barh(bot10_c["gloss"][::-1], bot10_c["f1"][::-1], color="#d62728")
axes[1].set_title(f"Top-10 Most Challenging Classes (F1) - {winning_exp}")
axes[1].set_xlabel("F1 Score")
axes[1].set_xlim(0, 1.05)

plt.tight_layout()
p_perclass = PLOTS_DIR / "05_class_performance.png"
plt.savefig(p_perclass, dpi=130)
plt.show()


## 6. Multimodal Representation Analysis: Joint Vision-Language t-SNE


In [ ]:
# 2D t-SNE of Video Kinematics & Text Prototypes in Joint Space
if "mm_pose_language_boosted" in test_embeddings:
    ve_test, te_vocab = test_embeddings["mm_pose_language_boosted"]
    # Select top 15 most frequent classes in test set for clean visualization
    class_counts = pd.Series(y_test).value_counts()
    top_cids = class_counts.head(12).index.tolist()

    mask = np.isin(y_test, top_cids)
    ve_sub = ve_test[mask]
    y_sub = y_test[mask]
    te_sub = te_vocab[top_cids]

    # Combine video and text embeddings for joint 2D t-SNE projection
    all_emb = np.concatenate([ve_sub, te_sub], axis=0)
    tsne = TSNE(n_components=2, perplexity=min(15, len(all_emb)-1), random_state=SEED, max_iter=1000)
    emb_2d = tsne.fit_transform(all_emb)

    n_vid = len(ve_sub)
    vid_2d = emb_2d[:n_vid]
    txt_2d = emb_2d[n_vid:]

    plt.figure(figsize=(11, 7))
    cmap = plt.get_cmap("tab20", len(top_cids))

    for idx, cid in enumerate(top_cids):
        c_mask = (y_sub == cid)
        col = cmap(idx)
        g_name = id_to_class[cid]
        # Video clip points
        plt.scatter(vid_2d[c_mask, 0], vid_2d[c_mask, 1], color=col, alpha=0.6, edgecolors="none", s=45)
        # Text prototype point
        plt.scatter(txt_2d[idx, 0], txt_2d[idx, 1], color=col, marker="*", s=250, edgecolors="black", linewidth=1.2, label=g_name)
        plt.text(txt_2d[idx, 0] + 0.5, txt_2d[idx, 1] + 0.5, g_name, fontsize=8, fontweight="bold")

    plt.title("Joint Kinematic-Language Embedding Space (Test Video Clips vs. MiniLM Text Prototypes)")
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="ASL Glosses (*=Text)", fontsize=8)
    plt.tight_layout()
    p_tsne = PLOTS_DIR / "05_tsne_multimodal_alignment.png"
    plt.savefig(p_tsne, dpi=130)
    plt.show()
    print(f"Saved multimodal t-SNE alignment plot: {p_tsne}")


## 7. Export Production Artifacts & Results Summary


In [ ]:
import json
import shutil

# 1. Update Unified experiments/results_summary.csv with test results
res_csv = EXP_DIR / "results_summary.csv"
if res_csv.is_file():
    old_df = pd.read_csv(res_csv)
    old_df = old_df[~((old_df["experiment_id"].isin(test_df["experiment_id"])) & (old_df["split"] == "test"))]
    frames = [df.dropna(axis=1, how="all") for df in (old_df, test_df) if not df.empty]
    merged_df = pd.concat(frames, ignore_index=True)
else:
    merged_df = test_df
merged_df.to_csv(res_csv, index=False)

# 2. Save comprehensive evaluation JSON report
report_data = {
    "evaluation_split": "test",
    "num_test_clips": len(y_test),
    "num_classes": NUM_CLASSES,
    "winning_model": winning_exp,
    "winning_metrics": {
        "top1_accuracy": test_results[winning_exp]["top1_accuracy"],
        "top5_accuracy": test_results[winning_exp]["top5_accuracy"],
        "top10_accuracy": test_results[winning_exp].get("top10_accuracy"),
        "macro_f1": test_results[winning_exp]["macro_f1"],
        "weighted_f1": test_results[winning_exp].get("weighted_f1"),
        "unique_predicted_classes": test_results[winning_exp]["unique_predicted_classes"],
    },
    "all_models_summary": test_df.to_dict(orient="records"),
    "top_confused_pairs": confused_df.head(20).to_dict(orient="records"),
    "strongest_classes": top10_c.to_dict(orient="records"),
    "weakest_classes": bot10_c.to_dict(orient="records"),
}
report_path = EXP_DIR / "test_evaluation_report.json"
with open(report_path, "w") as f:
    json.dump(report_data, f, indent=2)

# 3. Export production model bundle
import shutil
pose_dim_val = POSE_DIM if "POSE_DIM" in globals() else (P_test.shape[-1] if "P_test" in globals() else 349)
prod_bundle = {
    "winning_model_id": winning_exp,
    "class_to_id": class_to_id,
    "id_to_class": id_to_class,
    "num_classes": NUM_CLASSES,
    "feature_dim": pose_dim_val,
    "test_metrics": report_data["winning_metrics"],
}
if winning_exp in test_probabilities:
    np.save(PROD_DIR / "test_probabilities.npy", test_probabilities[winning_exp])

# Copy winning model checkpoint or ensemble config
if winning_exp == "ensemble_equal_three":
    shutil.copy2(CKPT_DIR / "ensemble_boosted" / "ensemble_config.json", PROD_DIR / "ensemble_config.json")
    for m_id in ("pose_transformer_cosine", "pose_bilstm_attn", "mm_pose_language_boosted"):
        m_ckpt = CKPT_DIR / m_id / "best_model.pt"
        if m_ckpt.is_file():
            shutil.copy2(m_ckpt, PROD_DIR / f"{m_id}.pt")
else:
    shutil.copy2(CKPT_DIR / winning_exp / "best_model.pt", PROD_DIR / "model.pt")

with open(PROD_DIR / "production_metadata.json", "w") as f:
    json.dump(prod_bundle, f, indent=2)

print("=" * 80)
print("NOTEBOOK 05 (LOCKED TEST-SET EVALUATION & ANALYSIS) COMPLETED")
print("=" * 80)
print(f"Test Report JSON        : {report_path}")
print(f"Updated Results CSV     : {res_csv}")
print(f"Comparison Chart        : {p_comp}")
print(f"Confusion Matrix        : {p_conf}")
print(f"Class Breakdown Chart   : {p_perclass}")
print(f"Production Model Bundle : {PROD_DIR}")
print("\nOfficial Test Summary:")
disp_cols = ["experiment_id", "top1_accuracy", "top5_accuracy", "macro_f1"]
if "v2t_recall@5" in test_df:
    disp_cols.append("v2t_recall@5")
if "v2t_mrr" in test_df:
    disp_cols.append("v2t_mrr")
print(test_df[disp_cols].to_string(index=False))
